# Reconocimiento de Actividad Humana mediante Teléfonos Inteligentes

Notebook con el código real del proyecto, celda por celda (no son solo llamadas a scripts).
Usa el dataset UCI HAR que ya viene incluido en el repositorio de GitHub, así que no hay que descargar nada aparte.

**Ejecuta las celdas en orden, de arriba hacia abajo.**

## 0. Preparar el entorno

In [ ]:
%cd /content
!rm -rf proyecto-en-es-es
!git clone https://github.com/sebastianmarinc19-sudo/proyecto-en-es-es.git
%cd proyecto-en-es-es

In [ ]:
# Colab ya trae pandas, numpy, scikit-learn, matplotlib, seaborn, scipy y
# joblib preinstalados y compatibles entre sí. Forzar --upgrade en todos
# rompe esa compatibilidad (sube pandas/numpy a versiones que chocan con
# otras librerías de Colab). Solo instalamos lo que falta: xgboost.
!pip install -q xgboost

In [ ]:
import sys
sys.path.insert(0, ".")  # para poder importar el paquete src/

import time
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

from src.datos import cargar_datos, guardar_reporte_limpieza, NOMBRES_ACTIVIDADES_ES, CARPETA_MODELOS, CARPETA_RESULTADOS, CARPETA_GRAFICOS, RUTA_NORMALIZADOR
from src.models import EntrenadorModelos
from src.evaluation import EvaluadorModelos, NOMBRES_METRICAS_ES

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

print("✓ Librerías y módulos del proyecto importados")

## 1. Cargar los datos

In [ ]:
datos = cargar_datos()

print(f"Train: {datos.entrenamiento.shape}")
print(f"Test:  {datos.prueba.shape}")
print(f"Características: {len(datos.nombres_caracteristicas)}")
print(f"Actividades: {len(datos.nombres_actividades)}")

In [ ]:
print("Muestras por actividad:\n")
for id_actividad, nombre_actividad in datos.nombres_actividades.items():
    cantidad_train = (datos.actividades_entrenamiento == id_actividad).sum()
    cantidad_test = (datos.actividades_prueba == id_actividad).sum()
    print(f"  {nombre_actividad:20} -> Train: {cantidad_train:4} | Test: {cantidad_test:4}")

print(f"\nVoluntarios -> Train: {len(np.unique(datos.voluntarios_entrenamiento))} | Test: {len(np.unique(datos.voluntarios_prueba))}")

## 2. Limpieza y verificación de datos

La limpieza (revisar valores faltantes, duplicados, etiquetas válidas y rango de valores) ya se aplicó dentro de `cargar_datos()`.
Aquí solo se imprime el reporte de lo que encontró.

In [ ]:
reporte = datos.reporte_limpieza

print(f"Valores faltantes (train): {reporte['faltantes_train']}")
print(f"Valores faltantes (test):  {reporte['faltantes_test']}")
print(f"Filas duplicadas (train):  {reporte['duplicados_train']}")
print(f"Filas duplicadas (test):   {reporte['duplicados_test']}")
print(f"Etiquetas válidas (train): {'Sí' if reporte['etiquetas_train_validas'] else 'No'}")
print(f"Etiquetas válidas (test):  {'Sí' if reporte['etiquetas_test_validas'] else 'No'}")
print(f"Rango de valores: [{reporte['valor_minimo']:.4f}, {reporte['valor_maximo']:.4f}]")

if reporte["dataset_estaba_limpio"]:
    print("\n✓ No se encontraron valores faltantes ni filas duplicadas.")
    print("  El dataset ya viene normalizado por el proveedor (UCI), no requiere limpieza adicional.")
else:
    for accion in reporte["acciones"]:
        print(f"⚠ {accion}")

ruta_reporte_limpieza = guardar_reporte_limpieza(reporte)
print(f"\n✓ Reporte guardado en: {ruta_reporte_limpieza}")

## 3. Análisis exploratorio (EDA) — gráficos

In [ ]:
CARPETA_GRAFICOS.mkdir(parents=True, exist_ok=True)

lista_actividades = datos.lista_actividades_en_espanol()
ids_actividades = sorted(datos.nombres_actividades.keys())
conteo_train = [np.sum(datos.actividades_entrenamiento == i) for i in ids_actividades]
conteo_test = [np.sum(datos.actividades_prueba == i) for i in ids_actividades]

In [ ]:
# Gráfico 1: distribución de actividades
fig, ejes = plt.subplots(1, 2, figsize=(14, 5))

posiciones = np.arange(len(lista_actividades))
ancho_barra = 0.35
ejes[0].bar(posiciones - ancho_barra/2, conteo_train, ancho_barra, label="Entrenamiento", alpha=0.8)
ejes[0].bar(posiciones + ancho_barra/2, conteo_test, ancho_barra, label="Prueba", alpha=0.8)
ejes[0].set_xlabel("Actividad")
ejes[0].set_ylabel("Número de Muestras")
ejes[0].set_title("Distribución de Actividades")
ejes[0].set_xticks(posiciones)
ejes[0].set_xticklabels(lista_actividades, rotation=45, ha="right")
ejes[0].legend()
ejes[0].grid(axis="y", alpha=0.3)

total_por_actividad = [conteo_train[i] + conteo_test[i] for i in range(len(lista_actividades))]
ejes[1].pie(total_por_actividad, labels=lista_actividades, autopct="%1.1f%%", startangle=90)
ejes[1].set_title("Proporción de Actividades")

plt.tight_layout()
plt.savefig(CARPETA_GRAFICOS / "01_distribucion_actividades.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Gráfico 2: distribución por voluntario
fig, ejes = plt.subplots(1, 2, figsize=(14, 5))

ids_train, cantidad_por_id_train = np.unique(datos.voluntarios_entrenamiento, return_counts=True)
ids_test, cantidad_por_id_test = np.unique(datos.voluntarios_prueba, return_counts=True)

ejes[0].bar(ids_train, cantidad_por_id_train, alpha=0.7, color="steelblue")
ejes[0].set_xlabel("ID Voluntario")
ejes[0].set_ylabel("Número de Muestras")
ejes[0].set_title("Distribución por Voluntario - Entrenamiento")
ejes[0].grid(axis="y", alpha=0.3)

ejes[1].bar(ids_test, cantidad_por_id_test, alpha=0.7, color="coral")
ejes[1].set_xlabel("ID Voluntario")
ejes[1].set_ylabel("Número de Muestras")
ejes[1].set_title("Distribución por Voluntario - Prueba")
ejes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(CARPETA_GRAFICOS / "02_distribucion_voluntarios.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Gráfico 3: estadísticas de características
fig, ejes = plt.subplots(2, 2, figsize=(14, 10))

ejes[0, 0].hist(datos.entrenamiento.mean(), bins=30, alpha=0.7, color="steelblue")
ejes[0, 0].set_xlabel("Media")
ejes[0, 0].set_ylabel("Frecuencia")
ejes[0, 0].set_title("Distribución de Medias por Característica")
ejes[0, 0].grid(alpha=0.3)

ejes[0, 1].hist(datos.entrenamiento.std(), bins=30, alpha=0.7, color="coral")
ejes[0, 1].set_xlabel("Desviación Estándar")
ejes[0, 1].set_ylabel("Frecuencia")
ejes[0, 1].set_title("Distribución de Desviación Estándar por Característica")
ejes[0, 1].grid(alpha=0.3)

ejes[1, 0].hist(datos.entrenamiento.min(), bins=30, alpha=0.7, color="green")
ejes[1, 0].set_xlabel("Valor Mínimo")
ejes[1, 0].set_ylabel("Frecuencia")
ejes[1, 0].set_title("Distribución de Mínimos por Característica")
ejes[1, 0].grid(alpha=0.3)

ejes[1, 1].hist(datos.entrenamiento.max(), bins=30, alpha=0.7, color="orange")
ejes[1, 1].set_xlabel("Valor Máximo")
ejes[1, 1].set_ylabel("Frecuencia")
ejes[1, 1].set_title("Distribución de Máximos por Característica")
ejes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(CARPETA_GRAFICOS / "03_estadisticas_caracteristicas.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Gráfico 4: boxplot de características seleccionadas
fig, ejes = plt.subplots(2, 3, figsize=(16, 10))
ejes = ejes.flatten()

caracteristicas_a_revisar = [0, 50, 100, 200, 300, 400]
for posicion, columna in enumerate(caracteristicas_a_revisar):
    valores_por_actividad = [datos.entrenamiento[datos.actividades_entrenamiento == i][columna]
                              for i in ids_actividades]
    ejes[posicion].boxplot(valores_por_actividad, labels=lista_actividades)
    ejes[posicion].set_title(f"Característica {columna}")
    ejes[posicion].set_ylabel("Valor")
    ejes[posicion].grid(alpha=0.3)
    plt.setp(ejes[posicion].xaxis.get_majorticklabels(), rotation=45, ha="right", fontsize=8)

plt.tight_layout()
plt.savefig(CARPETA_GRAFICOS / "04_boxplot_caracteristicas.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Gráfico 5: correlación entre las primeras 20 características
primeras_20_caracteristicas = datos.entrenamiento.iloc[:, :20]
matriz_correlacion = primeras_20_caracteristicas.corr()

fig, eje = plt.subplots(figsize=(12, 10))
sns.heatmap(matriz_correlacion, cmap="coolwarm", center=0, ax=eje, cbar_kws={"label": "Correlación"})
eje.set_title("Matriz de Correlación (Primeras 20 Características)")
plt.tight_layout()
plt.savefig(CARPETA_GRAFICOS / "05_correlacion_caracteristicas.png", dpi=300, bbox_inches="tight")
plt.show()

## 4. Normalización de los datos

In [ ]:
normalizador = StandardScaler()
entrenamiento_normalizado = normalizador.fit_transform(datos.entrenamiento)
prueba_normalizada = normalizador.transform(datos.prueba)

CARPETA_RESULTADOS.mkdir(parents=True, exist_ok=True)
joblib.dump(normalizador, RUTA_NORMALIZADOR)

print(f"Media train normalizado: {entrenamiento_normalizado.mean():.4f}")
print(f"Std train normalizado:   {entrenamiento_normalizado.std():.4f}")
print(f"✓ Normalizador guardado en: {RUTA_NORMALIZADOR}")

## 5. Entrenamiento de los 5 modelos

Cada modelo se entrena en su propia celda, para poder explicarlo por separado en la presentación.

In [ ]:
if "entrenador" not in globals():
    entrenador = EntrenadorModelos()
    entrenador.crear_modelos()
    resultados_por_modelo = {}
else:
    print("⚠ 'entrenador' ya existía, se reutiliza sin borrar los modelos ya entrenados.")

CARPETA_MODELOS.mkdir(parents=True, exist_ok=True)


def entrenar_y_evaluar(nombre_modelo):
    """Entrena un modelo, lo guarda en disco y mide su exactitud en train/test."""
    inicio = time.time()
    entrenador.entrenar_modelo(nombre_modelo, entrenamiento_normalizado, datos.actividades_entrenamiento)
    tiempo = time.time() - inicio

    ruta_modelo = CARPETA_MODELOS / f"{nombre_modelo.replace(' ', '_')}.pkl"
    entrenador.guardar_modelo(nombre_modelo, ruta_modelo)

    exactitud_train = (entrenador.predecir(nombre_modelo, entrenamiento_normalizado) == datos.actividades_entrenamiento).mean()
    exactitud_test = (entrenador.predecir(nombre_modelo, prueba_normalizada) == datos.actividades_prueba).mean()

    resultados_por_modelo[nombre_modelo] = {
        "exactitud_train": exactitud_train,
        "exactitud_test": exactitud_test,
        "tiempo": tiempo,
    }
    print(f"✓ {nombre_modelo:20} Train: {exactitud_train:.4f} | Test: {exactitud_test:.4f} | Tiempo: {tiempo:.2f}s")

In [ ]:
entrenar_y_evaluar("Regresión Logística")

In [ ]:
entrenar_y_evaluar("Bosque Aleatorio")

In [ ]:
entrenar_y_evaluar("SVM")

In [ ]:
entrenar_y_evaluar("XGBoost")

In [ ]:
entrenar_y_evaluar("Red Neuronal")

In [ ]:
faltantes = set(entrenador.modelos.keys()) - set(resultados_por_modelo.keys())
if faltantes:
    print(f"⚠ Todavía faltan por entrenar: {faltantes}. Corre sus celdas de entrenamiento antes de seguir.")

resumen_entrenamiento = pd.DataFrame(resultados_por_modelo).T.sort_values("exactitud_test", ascending=False)
resumen_entrenamiento

## 6. Evaluación de los modelos sobre el conjunto de prueba

In [ ]:
if "evaluador" not in globals():
    evaluador = EvaluadorModelos()
else:
    print("⚠ 'evaluador' ya existía, se reutiliza sin borrar las evaluaciones ya hechas.")


def evaluar(nombre_modelo):
    if nombre_modelo not in entrenador.modelos_entrenados:
        raise RuntimeError(
            f"'{nombre_modelo}' todavía no está entrenado. "
            f"Corre la celda de entrenamiento de ese modelo (sección 5) antes de evaluarlo, "
            f"o usa 'Entorno de ejecución -> Ejecutar todo' para correr el notebook completo en orden."
        )
    predicciones = entrenador.predecir(nombre_modelo, prueba_normalizada)
    probabilidades = entrenador.predecir_probabilidad(nombre_modelo, prueba_normalizada)
    metricas = evaluador.evaluar_modelo(nombre_modelo, datos.actividades_prueba, predicciones, probabilidades)
    print(f"{nombre_modelo:20} Accuracy: {metricas['Accuracy']:.4f} | F1-Score: {metricas['F1-Score']:.4f}")

In [ ]:
evaluar("Regresión Logística")

In [ ]:
evaluar("Bosque Aleatorio")

In [ ]:
evaluar("SVM")

In [ ]:
evaluar("XGBoost")

In [ ]:
evaluar("Red Neuronal")

In [ ]:
faltantes = set(entrenador.modelos.keys()) - set(evaluador.resultados.keys())
if faltantes:
    raise RuntimeError(
        f"Faltan evaluar estos modelos: {faltantes}. "
        f"Corre sus celdas de la sección 6 antes de comparar."
    )

tabla_comparacion = evaluador.comparar_modelos()
tabla_comparacion

## 7. Gráficos de evaluación

In [ ]:
# Gráfico 6: comparación de métricas por modelo
fig, eje = plt.subplots(figsize=(12, 6))

tabla_metricas = tabla_comparacion[["Accuracy", "Precision", "Recall", "F1-Score"]].sort_values("F1-Score", ascending=False)
tabla_metricas.rename(columns=NOMBRES_METRICAS_ES).plot(kind="bar", ax=eje, width=0.8)
eje.set_title("Comparación de Métricas por Modelo", fontsize=14, fontweight="bold")
eje.set_ylabel("Puntaje")
eje.set_xlabel("Modelo")
eje.set_ylim([0, 1.05])
eje.legend(loc="lower right")
eje.grid(axis="y", alpha=0.3)
plt.setp(eje.xaxis.get_majorticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.savefig(CARPETA_GRAFICOS / "06_comparacion_modelos.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Gráfico 7: ranking de modelos por F1-Score
fig, eje = plt.subplots(figsize=(10, 6))

puntajes_f1 = tabla_comparacion.sort_values("F1-Score", ascending=True)["F1-Score"]
colores_barras = ["green" if nombre == puntajes_f1.idxmax() else "steelblue" for nombre in puntajes_f1.index]
eje.barh(range(len(puntajes_f1)), puntajes_f1.values, color=colores_barras, alpha=0.8)
eje.set_yticks(range(len(puntajes_f1)))
eje.set_yticklabels(puntajes_f1.index)
eje.set_xlabel("Puntaje F1")
eje.set_title("Ranking de Modelos (Puntaje F1)", fontsize=14, fontweight="bold")
eje.set_xlim([0, 1])
eje.grid(axis="x", alpha=0.3)

for posicion, valor in enumerate(puntajes_f1.values):
    eje.text(valor + 0.02, posicion, f"{valor:.4f}", va="center")

plt.tight_layout()
plt.savefig(CARPETA_GRAFICOS / "07_ranking_modelos.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Gráfico 8: matrices de confusión de los 3 mejores modelos
mejores_modelos = tabla_comparacion.nlargest(3, "F1-Score").index

fig, ejes = plt.subplots(1, 3, figsize=(18, 5))

for posicion, nombre_modelo in enumerate(mejores_modelos):
    predicciones = entrenador.predecir(nombre_modelo, prueba_normalizada)
    matriz_confusion = confusion_matrix(datos.actividades_prueba, predicciones)

    sns.heatmap(matriz_confusion, annot=True, fmt="d", cmap="Blues", ax=ejes[posicion],
                cbar=False, xticklabels=range(1, 7), yticklabels=range(1, 7))
    ejes[posicion].set_title(f'{nombre_modelo}\nPuntaje F1: {tabla_comparacion.loc[nombre_modelo, "F1-Score"]:.4f}')
    ejes[posicion].set_ylabel("Verdadero")
    ejes[posicion].set_xlabel("Predicción")

plt.tight_layout()
plt.savefig(CARPETA_GRAFICOS / "08_confusion_matrices.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Gráfico 9: radar del mejor modelo
nombre_mejor_modelo = tabla_comparacion["F1-Score"].idxmax()
metricas_mejor_modelo = tabla_comparacion.loc[nombre_mejor_modelo, ["Accuracy", "Precision", "Recall", "F1-Score"]]

fig, eje = plt.subplots(figsize=(8, 8), subplot_kw=dict(projection="polar"))

angulos = np.linspace(0, 2 * np.pi, len(metricas_mejor_modelo), endpoint=False).tolist()
valores = metricas_mejor_modelo.values.tolist()
angulos += angulos[:1]
valores += valores[:1]

eje.plot(angulos, valores, "o-", linewidth=2, color="green", label=nombre_mejor_modelo)
eje.fill(angulos, valores, alpha=0.25, color="green")
eje.set_xticks(angulos[:-1])
eje.set_xticklabels([NOMBRES_METRICAS_ES[nombre] for nombre in metricas_mejor_modelo.index])
eje.set_ylim(0, 1)
eje.set_title(f"Métricas del Mejor Modelo\n{nombre_mejor_modelo}", fontsize=14, fontweight="bold", pad=20)
eje.grid(True)

plt.tight_layout()
plt.savefig(CARPETA_GRAFICOS / "09_radar_mejor_modelo.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"🏆 Mejor modelo: {nombre_mejor_modelo}")

## 8. Guardar resultados

In [ ]:
ruta_csv = CARPETA_RESULTADOS / "metricas_modelos.csv"
tabla_comparacion.to_csv(ruta_csv, index_label="Modelo")
print(f"✓ Métricas guardadas en: {ruta_csv}")

predicciones_mejor_modelo = entrenador.predecir(nombre_mejor_modelo, prueba_normalizada)
print(f"\nClasificación por actividad ({nombre_mejor_modelo}):\n")
print(classification_report(
    datos.actividades_prueba, predicciones_mejor_modelo,
    target_names=[datos.nombres_actividades[i] for i in sorted(datos.nombres_actividades.keys())]
))

## 9. Demo en vivo: predicción con el mejor modelo

Toma muestras al azar del conjunto de prueba y predice su actividad con el modelo de mejor F1-Score, mostrando la confianza de cada predicción.

In [ ]:
cantidad_muestras = 10
semilla = 42

total_muestras = len(datos.prueba)
generador_aleatorio = np.random.RandomState(semilla)
indices = generador_aleatorio.choice(total_muestras, size=cantidad_muestras, replace=False)

filas_reporte = []
for indice in indices:
    muestra = prueba_normalizada[indice].reshape(1, -1)
    actividad_real_id = datos.actividades_prueba[indice]

    prediccion_id = entrenador.predecir(nombre_mejor_modelo, muestra)[0]
    probabilidades = entrenador.predecir_probabilidad(nombre_mejor_modelo, muestra)
    confianza = probabilidades[0].max() if probabilidades is not None else float("nan")

    nombre_real = NOMBRES_ACTIVIDADES_ES[actividad_real_id]
    nombre_prediccion = NOMBRES_ACTIVIDADES_ES[prediccion_id]
    acerto = actividad_real_id == prediccion_id

    filas_reporte.append({
        "indice": int(indice),
        "actividad_real": nombre_real,
        "prediccion": nombre_prediccion,
        "confianza (%)": round(float(confianza) * 100, 1),
        "acertó": "Sí" if acerto else "No",
    })

tabla_demo = pd.DataFrame(filas_reporte)
aciertos = (tabla_demo["acertó"] == "Sí").sum()
print(f"Modelo usado: {nombre_mejor_modelo}")
print(f"Exactitud en esta demo: {aciertos}/{len(tabla_demo)} ({aciertos/len(tabla_demo)*100:.1f}%)\n")
tabla_demo

---
## Notas

- Este notebook usa las mismas clases y funciones de `src/` (mismo código verificado y probado en local), solo que la orquestación está escrita celda por celda en vez de ocultarse dentro de un solo script.
- Cada vez que reinicies el entorno de Colab (o pase mucho tiempo inactivo), hay que volver a correr las celdas desde el paso 0, porque Colab no guarda archivos entre sesiones.
- Para repetir un ejemplo exacto en la presentación, cambia `semilla` o usa un índice fijo con `entrenador.predecir(nombre_mejor_modelo, prueba_normalizada[indice].reshape(1, -1))`.